# | default_exp Testing Polars for EDA, migrating cleaned data to Vespa

In [ ]:
# | hide
# import adbc_driver_postgresql.dbapi
# from datetime import datetime
# from enum import Enum
import json
import polars as pl
# import requests
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
# import vespa
from vespa.deployment import VespaDocker
from vespa.io import VespaResponse, VespaQueryResponse
from vespa.package import RankProfile, Function, FirstPhaseRanking

In [ ]:
import os

os.environ['CUDA_LAUNCH_BLOCKING']= "1"
# os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
# make sure postgres is running!
# load postgres credentials
postgres_key_path = "../secrets/postgres_login.json"
with open(postgres_key_path, "r") as fo:
    postgres_key = json.loads(fo.read())
    user = postgres_key["user"]
    password = postgres_key["password"]
    host = postgres_key["host"]


In [ ]:
# send dataframe back into Postgres
cleaned_data_uri = f"postgresql://{user}:{password}@{host}/mealeon"

query = """
    SELECT
        *
    FROM cleaned_recipes
"""

df = pl.read_database_uri(query, cleaned_data_uri, engine="adbc")

In [ ]:
print(df)

shape: (128_736, 11)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ mealeon_i ┆ language ┆ source_id ┆ title     ┆ … ┆ photo_url ┆ descripti ┆ steps     ┆ cuisines  │
│ d         ┆ ---      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ on        ┆ ---       ┆ ---       │
│ ---       ┆ str      ┆ str       ┆ str       ┆   ┆ str       ┆ ---       ┆ list[str] ┆ list[str] │
│ str       ┆          ┆           ┆           ┆   ┆           ┆ str       ┆           ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ AfricanBi ┆ English  ┆ 629883    ┆ Smoked    ┆ … ┆ https://w ┆ Enjoy fal ┆ ["Remove  ┆ ["US Sout │
│ tes-3f1a4 ┆          ┆           ┆ Spatchcoc ┆   ┆ ww.africa ┆ l-off-the ┆ the       ┆ hern"]    │
│ fc7e09937 ┆          ┆           ┆ k Turkey  ┆   ┆ nbites.co ┆ -bone     ┆ giblet    ┆           │
│ 5ad…      ┆          ┆           ┆           ┆   ┆ m/w…      ┆ goodn

### Try mixing in PyVespa

Import model, now enabled in SentenceTransformers

In [ ]:
from sentence_transformers import SentenceTransformer
# from FlagEmbedding import BGEM3FlagModel

model = SentenceTransformer("BAAI/bge-m3")
# model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

In [ ]:
# noticed nulls and empty lists...
df_non_null = df.with_columns(
    pl.col("ingredients").is_not_null()
    .alias("ingredients_not_null")
)

df_non_null.shape

(128736, 12)

In [ ]:
df_filtered = df_non_null.filter(
    (pl.col("ingredients").list.len() > 0)
    |
    (pl.col("ingredients_not_null") == True)
)

In [ ]:
df_filtered.shape

(128636, 12)

In [ ]:
# encode ingredients
df_string = df_filtered.with_columns(
    pl.col("ingredients").list.join(" || ")
    .alias('ingredients_string')
)


In [ ]:
ingredients_string = df_string['ingredients_string'].to_list()

In [ ]:
bge_kwargs = {'return_dense':True, 
              'return_sparse':True, 
              'return_colbert_vecs':True
              }

ingredient_embeddings = model.encode(
    ingredients_string,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/4020 [00:00<?, ?it/s]

In [ ]:
null_count = 0
for elem in df['ingredients_string'].to_list():
    if elem:
        continue
    else:
        null_count += 1
print(null_count)

102


In [ ]:
df.filter(pl.col("ingredients").list.len() == 0)

mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines,ingredients_string
str,str,str,str,str,str,list[str],str,str,list[str],list[str],str
"""AllRecipes-f8678701d7711e5388b…","""English""","""245838""","""Italian Sausage Stuffed Mushro…","""AllRecipes""","""https://www.allrecipes.com/rec…",[],"""Missing Photo""","""These simple stuffed mushrooms…",[],"[""Missing Cuisine""]",""""""
"""AllRecipes-7d4e61bffeb93775df7…","""English""","""211045""","""PIZZA BURGERS""","""AllRecipes""","""https://www.allrecipes.com/rec…",[],"""Missing Photo""","""A popular Italian twist to an …",[],"[""Missing Cuisine""]",""""""


In [ ]:
df.with_columns(pl.col("ingredients").is_null()).select(["mealeon_id", "title", "ingredients", "ingredients_string"])

mealeon_id,title,ingredients,ingredients_string
str,str,bool,str
"""AfricanBites-3f1a4fc7e099375ad…","""Smoked Spatchcock Turkey""",false,"""salt and pepper for seasoning …"
"""AfricanBites-ad9870f6689604624…","""How to Brine a Turkey""",false,"""1 gallon water || 1 gallon app…"
"""AfricanBites-9b119ba4403faa097…","""Refried Beans""",false,"""6 tbsp lard, bacon drippings, …"
"""AfricanBites-378e4cd9d9469919c…","""Lemon Blueberry Scones""",false,"""2½ cups (312.5 g) all-purpose …"
"""AfricanBites-b81cc0ca56e2dc4ea…","""Chocolate Pecan Pie""",false,"""1 9-inch pie crust (deep-dish…"
…,…,…,…
"""AllRecipes-03674130873d5db1657…","""Seared Scallops with Pineapple…",false,"""1 shallot, sliced crosswise |…"
"""AllRecipes-78da6627346b728a325…","""Double Batch Caramel Brownies""",false,"""1 cup semisweet chocolate chip…"
"""AllRecipes-1ed148bf72e4ba56cdb…","""Beet and Fennel Salad with Goa…",false,"""2 red beets || 2 golden beet…"


In [ ]:
df.filter((pl.col("ingredients_string").str.len_chars() == 0) or (pl.col("ingredients_string").is_null()))

mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines,ingredients_string
str,str,str,str,str,str,list[str],str,str,list[str],list[str],str
"""AllRecipes-f8678701d7711e5388b…","""English""","""245838""","""Italian Sausage Stuffed Mushro…","""AllRecipes""","""https://www.allrecipes.com/rec…",[],"""Missing Photo""","""These simple stuffed mushrooms…",[],"[""Missing Cuisine""]",""""""
"""AllRecipes-7d4e61bffeb93775df7…","""English""","""211045""","""PIZZA BURGERS""","""AllRecipes""","""https://www.allrecipes.com/rec…",[],"""Missing Photo""","""A popular Italian twist to an …",[],"[""Missing Cuisine""]",""""""


In [ ]:
ingredient_embeddings

<tqdm.std.tqdm>

Following documentation [here](https://pyvespa.readthedocs.io/en/latest/getting-started-pyvespa.html)

In [ ]:
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
    DocumentSummary,
    Summary
)

vespa_name = "mealeon3"
vespa_port = 8183
vespa_cfgsrv_port = 19092


package = ApplicationPackage(
    name=vespa_name,
    schema=[
        Schema(
            name=vespa_name,
            document=Document(
                fields=[
                    Field(
                        name="language", 
                        type="string", 
                        indexing=["set_language"],
                    ),
                    Field(
                        name="id",
                        type="string",
                        indexing=["index","summary"],
                        match=["exact"],
                    ),
                    Field(
                        name="title",
                        type="string",
                        indexing=["index", "summary"],
                        index="enable-bm25",
                        match=["word"],
                        bolding=True, 
                    ), 
                    Field(
                        name="description",
                        type="string",
                        indexing=["index"],
                        index="enable-bm25",
                        match=["text"],
                    ),                 
                    Field(
                        name="ingredients",
                        type="array<string>",
                        indexing=["index", "attribute"],
                        index="enable-bm25",
                        bolding=True,
                    ),
                    Field(
                        name="steps",
                        type="array<string>",
                        indexing=["index"],
                    ),
                    Field(
                        name="cuisines",
                        type="array<string>",
                        indexing=["attribute", "summary"],
                        match=["word"],
                        rank="filter"
                    ),
                    # Field(
                    #     name="embedding",
                    #     type="tensor<float>(x[384])",
                    #     indexing=[
                    #         'input title . " " . input body',
                    #         "embed",
                    #         "index",
                    #         "attribute",
                    #     ],
                    #     ann=HNSW(distance_metric="angular"),
                    #     is_document_field=False,
                    # ),
                    # borrowing from PyVespa documentation with BGE-M3 https://vespa-engine.github.io/pyvespa/examples/mother-of-all-embedding-models-cloud.html
                    # only ingredients
                    Field(
                        name="ingredients_lexical_rep",
                        type="tensor<bfloat16>(t{})",
                        indexing=["attribute"],
                    ),
                    Field(
                        name="ingredients_dense_rep",
                        type="tensor<bfloat16>(x[1024])",
                        indexing=["attribute"],
                        attribute=["distance-metric: angular"],
                    ),
                    Field(
                        name="ingredients_colbert_rep",
                        type="tensor<bfloat16>(t{}, x[1024])",
                        indexing=["attribute"],
            ),
                ]
            ),
            # fieldsets need to have compatible types and match settings
            fieldsets=[
            #     FieldSet(
            #         name="default", 
            #         fields=["title", "ingredients"]
            #     )
                FieldSet(
                    name="steps_ingreds",
                    fields=["steps", "ingredients"]
                )
            ],
            document_summaries=[
                    DocumentSummary(
                    name="mealeon_summary",
                    summary_fields=[
                        Summary("id"),
                        Summary("title"),
                        Summary("cuisines")
                    ]
                ),
            ],
            rank_profiles=[
                RankProfile(
                    name="default",
                    first_phase="nativeRank(title, ingredients)"
                ),
                RankProfile(
                    name="bm25",
                    inherits="default",
                    first_phase="bm25(title) + bm25(ingredients)",
                    # inputs=[("query(q)", "tensor<float>(x[384])")],
                    functions=[
                        Function(name="bm25sum", expression="bm25(title) + bm25(ingredients)")
                    ],
                ),
                RankProfile(
                    name="combined", 
                    inherits="default",
                    first_phase="bm25(title) + bm25(ingredients) + nativeRank(title) + nativeRank(ingredients)",
                    functions=[
                        Function(
                            name="bm25nativeRank",
                            expression="bm25(title) + bm25(ingredients) + nativeRank(title) + nativeRank(ingredients)")
                    ]
                ),
                RankProfile(
                    name="m3hybrid",
                    inputs=[
                        # linter complains about missing third value in tuple, but that's optional https://github.com/vespa-engine/pyvespa/issues/676                        
                        ("query(q_ingredients_dense)", "tensor<bfloat16>(x[1024])"),
                        ("query(q_ingredients_lexical)", "tensor<bfloat16>(t{})"),
                        ("query(q_ingredients_colbert_rep)", "tensor<bfloat16>(qt{}, x[1024])"),
                        ("query(q_ingredients_len_colbert)", "float"),
                    ],
                    functions=[
                        Function(
                            name="dense",
                            expression="cosine_similarity(query(q_ingredients_dense), attribute(ingredients_dense_rep),x)",
                        ),
                        Function(
                            name="lexical", 
                            expression="sum(query(q_ingredients_lexical) * attribute(ingredients_lexical_rep))"
                        ),
                        Function(
                            name="max_sim",
                            expression="""
                                sum(
                                    reduce(
                                        sum(
                                            query(q_ingredients_colbert_rep) * attribute(ingredients_colbert_rep) , x
                                        ),
                                        max, t
                                    ),
                                    qt
                                )/query(q_ingredients_len_colbert)
                            """,
                        ),
                    ],
                    first_phase=FirstPhaseRanking(
                        expression="0.4*dense + 0.2*lexical +  0.4*max_sim", rank_score_drop_limit=0.0
                    ),
                    match_features=["dense", "lexical", "max_sim", "bm25(ingredients)"],
                )
                # RankProfile(
                #     name="semantic",
                #     inputs=[("query(q)", "tensor<float>(x[384])")],
                #     first_phase="closeness(field, embedding)",
                # ),
                # RankProfile(
                #     name="fusion",
                #     inherits="bm25",
                #     inputs=[("query(q)", "tensor<float>(x[384])")],
                #     first_phase="closeness(field, embedding)",
                #     global_phase=GlobalPhaseRanking(
                #         expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                #         rerank_count=1000,
                #     ),
                # ),

            ],
        )
    ],
    components=[
        # Component(
            # id="bge-m3",
            # type="hugging-face-embedder",
            # parameters=[
            #     Parameter(
            #         "transformer-model",
            #         {
            #             # "url": "https://huggingface.co/BAAI/bge-m3/blob/main/onnx/model.onnx",
            #             "url": "https://huggingface.co/BAAI/bge-m3/resolve/main/onnx/model.onnx", #direct download
            #             # "path": "models/onnx/bge-m3/model.onnx",
            #         },
            #     ),
            #     Parameter(
            #         "tokenizer-model",
            #         {
            #             # "url": "https://huggingface.co/BAAI/bge-m3/blob/main/onnx/tokenizer.json",
            #             "url": "https://huggingface.co/BAAI/bge-m3/resolve/main/onnx/tokenizer.json",
            #             # "path": "models/onnx/bge-m3/tokenizer.json",
            #         },
            #     ),
            #     # Parameter(
            #     #     "normalize", True
            #     # ),
            #     # Parameter(
            #     #     "pooling-strategy", "mean"
            #     # )
            # ],
            # )
        Component(
            id="bge-m3",
            
            config = {
                "name":"ai.vespa.llm.clients.llm-local-client",
                "class":"ai.vespa.llm.clients.LocalLLM",
                "model":{"url": "https://huggingface.co/CronoZero15/mealeon/blob/main/models/bge-m3/bge-m3.gguf"}
            }
            
            # parameters = [
            #     # Parameter(
            #     #     "class",
            #     #     "ai.vespa.llm.clients.LocalLLM"
            #     # ),
            #     # Parameter(
            #     #     "name",
            #     #     "ai.vespa.llm.clients.llm-local-client"
            #     # ),
            #     Parameter(
            #         "model",
            #         {"url": "https://huggingface.co/CronoZero15/mealeon/blob/main/models/bge-m3/bge-m3.gguf"}
            #     ),
            # ]
        )
    ],
)

In [ ]:
# try mixing in PyVespa
vespa_docker = VespaDocker(port=vespa_port,
                           cfgsrv_port=vespa_cfgsrv_port)
app = vespa_docker.deploy(application_package=package)


Container.com.yahoo.jdisc.core.StandaloneMain    JDisc exiting: Throwable caught: exception=com.yahoo.container.di.componentgraph.core.ComponentNode$ComponentConstructorException: Error constructing 'bge-m3' of type 'ai.vespa.embedding.huggingface.HuggingFaceEmbedder': nullCaused by: java.lang.RuntimeException: ONNX Runtime exception\n\tat ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:161)\n\tat ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:156)\n\tat ai.vespa.modelintegration.evaluator.OnnxEvaluator.<init>(OnnxEvaluator.java:36)\n\tat ai.vespa.modelintegration.evaluator.OnnxRuntime.evaluatorOf(OnnxRuntime.java:81)\nCaused by: ai.onnxruntime.OrtException: Error code - ORT_RUNTIME_EXCEPTION - message: Exception during initialization: filesystem error: cannot get file size: No such file or directory [/opt/vespa/var/db/vespa/download/5255768676247164915/model.onnx_data]\n\tat ai.onnxruntime.OrtSession.createSession(Native Method)\n\tat ai.onnxruntime.OrtSession.<init>(OrtSession.java:74)\n\tat ai.onnxruntime.OrtEnvironment.createSession(OrtEnvironment.java:236)\n\tat ai.onnxruntime.OrtEnvironment.createSession(OrtEnvironment.java:221)\n\tat ai.vespa.modelintegration.evaluator.OnnxRuntime$1.create(OnnxRuntime.java:46)\n\tat ai.vespa.modelintegration.evaluator.OnnxRuntime.acquireSession(OnnxRuntime.java:149)\n\tat ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:144)\n\t... 3 more\n

Container.com.yahoo.jdisc.core.StandaloneMain    JDisc exiting: Throwable caught: \nexception=\ncom.yahoo.container.di.componentgraph.core.ComponentNode$ComponentConstructorException: Error constructing 'bge-m3' of type 'ai.vespa.embedding.huggingface.HuggingFaceEmbedder': null\nCaused by: java.lang.RuntimeException: expected value at line 1 column 1\n\tat ai.djl.huggingface.tokenizers.jni.TokenizersLibrary.createTokenizerFromString(Native Method)\n\tat com.yahoo.language.huggingface.HuggingFaceTokenizer.lambda$getModelInfo$6(HuggingFaceTokenizer.java:125)\n\tat com.yahoo.language.huggingface.HuggingFaceTokenizer.withContextClassloader(HuggingFaceTokenizer.java:150)\n\tat com.yahoo.language.huggingface.HuggingFaceTokenizer.getModelInfo(HuggingFaceTokenizer.java:122)\n

Container.com.yahoo.jdisc.core.StandaloneMain    JDisc exiting: Throwable caught: 
exception=
com.yahoo.container.di.componentgraph.core.ComponentNode$ComponentConstructorException: Error constructing 'bge-m3' of type 'ai.vespa.embedding.huggingface.HuggingFaceEmbedder': null
Caused by: java.lang.RuntimeException: ONNX Runtime exception
    at ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:161)
        at ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:156)
            at ai.vespa.modelintegration.evaluator.OnnxEvaluator.<init>(OnnxEvaluator.java:36)
                at ai.vespa.modelintegration.evaluator.OnnxRuntime.evaluatorOf(OnnxRuntime.java:81)
                Caused by: ai.onnxruntime.OrtException: Error code - ORT_RUNTIME_EXCEPTION - message: Exception during initialization: filesystem error: cannot get file size: No such file or directory [/opt/vespa/var/db/vespa/download/5255768676247164915/model.onnx_data]
                    at ai.onnxruntime.OrtSession.createSession(Native Method)
                        at ai.onnxruntime.OrtSession.<init>(OrtSession.java:74)
                            at ai.onnxruntime.OrtEnvironment.createSession(OrtEnvironment.java:236)
                                at ai.onnxruntime.OrtEnvironment.createSession(OrtEnvironment.java:221)
                                    at ai.vespa.modelintegration.evaluator.OnnxRuntime$1.create(OnnxRuntime.java:46)
                                        at ai.vespa.modelintegration.evaluator.OnnxRuntime.acquireSession(OnnxRuntime.java:149)
                                            at ai.vespa.modelintegration.evaluator.OnnxEvaluator.createSession(OnnxEvaluator.java:144)\n\t... 3 more\n

In [ ]:
import requests 
hf_response = requests.get("https://huggingface.co/BAAI/bge-m3/blob/main/onnx/tokenizer.json")

hf_response.status_code

In [ ]:
hf_response.headers

In [ ]:
hf_response.text

In [ ]:
# query should be recipe name?
    # WHERE title !contains {query}
# cuisine name should be in the WHERE filter clause of YQL
    # AND WHERE cuisine NOT IN {cuisines}
# how to penalize similar title?

# start with plain keyword search

with app.syncio(connections=1) as session:
    query = "Buffalo Wings"
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources {vespa_name} where (title contains '{query}') limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()

In [ ]:
print(response.hits)

In [ ]:
import pprint

pprint.pp(response.hits)

In [ ]:
# list of recipe IDs
baseline_to_search = [
    'BBCFood-11015de4db20e53dcda0035673dfdbeca355d2a1d37d71f13d5fe5ea7ccca02d',
    'Epicurious-32b7b2040aba4e44b0f5e34b094aafc6b1b7168e689ba6c92428fa8c9e235c5a',
    'Epicurious-9e24d01c507b4ee64ee484fc4efafb5f3d44709abb8fc36f1b34a91f7b69b678',
    'AllRecipes-fddb273d401ac45b568695daeb1abe0b632696dd867992fd401e31dc794d66f9',
    'AllRecipes-715d84d02e183deaaf2400f7526c80a2b01a25032c1abb125c28fe4d879cc672', 
    'AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e', 
    'Epicurious-85d1faf0a617f34521543d7c64122288b8ba65ae55b7a37a9abeb22d8845ff11',
    'AllRecipes-867b4f3b539b381fb9f35252402fc14efc9d6fcc64353089959ac159372a8199', 
    'BBCFood-c6b9581aa49ee6b0cc3f686f58cfd124c395284cc46dfcde5bc931982975ee15',
    'BBCFood-d59ab715a7f6f0022155b1149e4e059b0497fd790993fb11f1795b901cae8f97',
    'AllRecipes-d431cefd9f595be8a8fcd7ff9e05a82ca1b79220c72edde1aa2a0b6574060c04',
    'BBCFood-056f016b0b0d8c383c1a6bf47077a46a64529c04ab043c14dce3b23d3980e0ed',
    'AllRecipes-d4bb95742a7fd71eb848a7d4b2a4dc13220e5bb378e1b6805d5ea493e0e1166d',
    'Epicurious-0293b95bfc74cf90dc07e2ee399312e40e03d14dd99e5db84e678a1393b131cc',
    'Panlasang_Pinoy-1d186c4a4c274e42580488e652e3448aa9942642fa9e990d89d3e1894056ce78',
    'AllRecipes-9bfd691bb0521c3bba0275d1ef18ac30ea15c2aeb0a48f2b9df1d5defd962bed',
    'Epicurious-e7efaeb37ea032d7faaad76c1e0675f987aa5eb162e5447927ae510c0301807e',
    'Epicurious-ca2d5e56121ae04e8b7bdcd07648796b2d95afed017fc56200535ad43ca8e51d',
    'Epicurious-7761b0f03aaf56b52b1de417213b66fddf1d49c554dabaacb23c9c7a260982ba',
    'Epicurious-e367b68ed6df8b0bccbeb73f9d82897294ef267fa3a74d30505ce7470b2384dc',
    'AllRecipes-a0ae2dfd0ca25ae3146df101d1dc0e7c54e39eec69617ff634f42d80d7b70be4',
    'AfricanBites-8e8956de6dd17820c4e5146820972193f2587d97e80ee1a754c92dce048e4f9e',
    'BBCFood-3aa9507e04707431ec512662444ef602910c6c0ecc997de7ea206729985c5c08',
    'AllRecipes-6914cfedf5d3af52ea66433959c958821b8940ee1948fbe29f841d48ad1de4a4',
    'AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23',
    'Epicurious-0a033ff3220eae4b638fd08de58646ba84ed5f184b89d792afe2ed90a8c4a859',
    'BBCFood-adcbfceaa8636f83ff8866d44bc7a44385c71250d73d427c08dbe345e4127143',
    'AllRecipes-81a545cc3eac6e8b89274a6a9f4a6574232e40c3917f499a431d9fa6ebc540bf',
    'BBCFood-0a7c810d7b7c99ae728f9a2deff14340edb099949f20fed75a8083c29ca83aab',
    'Epicurious-a0ab4c8e25f1fe8e65f400f93db23cf55e1e46cb4361cd65426f908f3013f33d',
    'AllRecipes-bb6dd1b85cfd3cbfeb68160c187862ec6de84739123733ceeaae2976141f8936',
    'Epicurious-3b409b8e753718bdb272130405491c4dcb4c26e73412b54cea408319dc932c6e',
    'AllRecipes-92c5c30f5c0b1655a1e1900fcdaa603917f2abf62529c130713412a844046e33',
    'AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0',
    'AllRecipes-384fd2f67b06a5bfdf532f94d5e89df99e08b91c5d9dace20fd29bba7b8fdad3',
    'Epicurious-aa7806301b6fc904134d692a2a425d0a0fd5a1f6f95cb431e8c31173db7d99fa',
    'AllRecipes-d9b8ecb976729402694fa448a53b741404ef3e10f6dd9fe2219fc6fb3fb7b038',
    'BBCFood-ee117ce84728c9b2e90654f59895a89dc7c10599b61bed83f55e9bdf723e6890',
    'BBCFood-53dc7bd8da3ef8a7dedfef0b2be1880f71585752d8113313c6d9e49bd5cba6fa',
    'BBCFood-2ab22cd0a3889f38cf740040d33aae5a4baa8568b70cde5233d6f8a673958140',
    'Epicurious-1c0fe6496b26a8b7bdf549e9ed70bc9802d7df404b89f1d3de6d01a1a44d31f8',
    'BBCFood-47410dc7097b2609d5c0a80e4ce6f938b00be18b52907e5a5322e33d6edee335',
    'AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e',
    'Epicurious-ca99be627e88806d8357a5cbfdd314b3643028f275e6ae1c90adfcf3ae9e30d4',
    'Epicurious-7e9ff3fc13b040941ed9f7a4bbfd809e17a9ce7d5b569f1f6c4ba96e9a9ae140',
    'Epicurious-4b6f818877199d375e801cec974c6f6b037b4c0b2aa2bbec0238dd77f797264d',
    'AllRecipes-c6d0cd77e37ed8069f0d9e72f9e832026459273ac1bc2587800dc8fcd1418ec7',
    'AllRecipes-48bde26e015dc9cbcbb540c4710248d538227d4afe5aa3e977e34710289789bb',
    'AllRecipes-cac0d29872bd9f49d34d8c9eac41a3806929420f84e36b8e579c3bce939cdc7b',
    'AllRecipes-9a5dc3fa2511284deeb5ee469d110874b0de9c021439cd0e088d3ef4f53f28bf',
    'BBCFood-6d6cc18dadaffa5ea10194b59aa6e310d34b1f713756d0b528cc57c5e61d92e3',
    'AllRecipes-5fc2fc72d7caf510b4ccde625c54a0907de1489d323d385d74c2a6c9c5d61c7b',
    'AfricanBites-56e7fc4d3a1db457444cf4744911356514b3ac961f04976a0fb127f0ac8145be',
    'AllRecipes-8aa892f3f4d827f781d2bc1491351d982144e2fa2bb3226ebf5cc6af6df5ac5a',
    'AllRecipes-e13da6adb836dcf514a8f285e162a9529271a647a11f2da6e60697541260c1ec',
    'Epicurious-aef6e4737b2b9248ae51893ec6d2295ab2380bc67e817ebef13416c65d44a3a6',
    'AllRecipes-e11c702c9c5f8d5c0d63a6b58a60d90b87f1f4e88dad3586b91b3248bb721504',
    'Epicurious-b6ae682f94cb0c4bf256ebfc569550702ef82612457ff8e601cf9e61e0b5f4ab',
    'Epicurious-9908959a4069c75bd1d95c4b11678d15a63a153c91d164d3f7d1159d35ae05dc',
    'AllRecipes-4f31ce862b67de4659f76e3a4de016a32770c878544d112aae95cc09b63d799a',
    'AllRecipes-f3aafa676ce976137a1e9a62a6beeca6288c2a4b0c35ba2c4da9cd6a4fa9f466',
    'Epicurious-b3f8d44e48bba1b47e5cf1482d8c5e076359fa919061bbfd2f075d131aea2025',
    'Epicurious-6a7b3ef6808832fbf1ab5a8e539eb794866d630c1b65218ee7b5c385b78f65a4',
    'AllRecipes-0caa9e3e35dee2910e6f81c869d1176a2de129b8e3c7fddf6e469b9e3977fadb',
    'AllRecipes-3c9cbb4d78d017a21a48907adbc31c2593ebbd0717b9391779a2f75b531fcbe4',
    'AllRecipes-3e906cbcfdee7e4c52d087efc24aea9c7e3b760a8c5f0d1c85c459f3354b643d',
    'Panlasang_Pinoy-b5a6e813375f4cbb399a2e68dec8f9448d64c4f01f2e5b5accd9188509875f17',
    'Epicurious-d2b862c91c84a69b9b2f2be1a1cc9702a2d6af8a6f1e5dcaccbacda6ed9f31b0',
    'AllRecipes-9e94827f263b0409c124dd60e95e4a61e1d446940e3ad3b88b4e3b43d00b6f65',
    'AllRecipes-b8116cc920cd9d916cd196927ad2af1062785cdaa3d3b4fc3e997ee86fd75c84',
    'Epicurious-4284e53129c7dc89f97f2f1933843946745df061370bd1ab738bf2afe1c537c3',
    'AllRecipes-2b9df9cae8603bc9f0c5ddadf04fe529c1174d38a7e32031e42ee9897ea8e05f',
    'AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72',
    'AllRecipes-bd864502bd1b052f501d3148d66effc966246a69a5e705c585fa366d9599e090',
    'AllRecipes-1b1ac4f279d0cbb0c680da7a1839d0031a52f3cc73681dbb2644cf96d6aec4f9'
]

In [ ]:
filtered_to_investigate = df.filter(pl.col("mealeon_id").is_in(baseline_to_search))

In [ ]:
filtered_to_investigate

First things we need to do are check that these recipes "make sense"
- Are these recipes good baselines?

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "ingredients", "description", "cuisines"])[0:5])

Let's re-apply the previous text processing on ingredients to shorten these huge lists and make them easier to read (previous text processing removed numbers and I think units)

Previously, using `TFIDFVectorizer`, the params were:
```
TfidfVectorizer(
    stop_words=stopwords_list,
    min_df=2,
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b",
    preprocessor=lemmatizer.lemmatize,
)
```

Stopwords were in a CSV located at `../food_stopwords.csv` and did include units and brands

Probably do not need to lemmatize, just apply that regex from token_pattern and filter out stopwords.

How to reapply in Polars?
- [X] Join list of strings (ingredients)
- [X] filter string based on regex and stopwords

In [ ]:
filtered_to_investigate.select("ingredients").to_series().list.join(separator="||")

In [ ]:
filtered_to_investigate = filtered_to_investigate.with_columns(
    pl.col("ingredients")
    .list.join(separator="BREAK")
    .str.extract_all(
        r"(?u)\b[a-zA-Z]{2,}\b"
    )
    .alias("regex_filtered_ingredients")
)

In [ ]:
filtered_to_investigate.glimpse()

In [ ]:
# lets load the food_stopwords.csv into a polars dataframe and extract into a set
with open("../food_stopwords.txt") as f:
    for line in f:
        food_stopwords = line.replace('"', '').split(",")

food_stopwords

In [ ]:
filtered_to_investigate = filtered_to_investigate.with_columns(
    pl.col("ingredients")
    .list.join(separator="|")
    .str.to_lowercase()
    .str.extract_all(
        r"(?u)\b[a-zA-Z]{2,}\b"
    )
    .list.set_difference(food_stopwords)
    .alias("regex_filtered_ingredients")
)

In [ ]:
filtered_to_investigate.select("regex_filtered_ingredients").glimpse()

Look at recipes, see if ingredients make sense with a canonical version online

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[0:5])


In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[5:10])


In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=200,
               set_tbl_width_chars=50
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[10:15])


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[15:20])


| title (str) | mealeon_id (str) | reference_url (str) | passable (y/n) |
│ Banana Pudding | AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e | https://www.the-girl-who-ate-everything.com/magnolia-bakerys-famous-banana-pudding/ | yes |
| Tiramisu | AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e | 
https://cooking.nytimes.com/recipes/1018684-classic-tiramisu | yes | 
| Apple Pie | AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0 | https://www.allrecipes.com/recipe/12682/apple-pie-by-grandma-ople/ | yes | 
│ Fish and Chips | AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23 | https://www.thespruceeats.com/best-fish-and-chips-recipe-434856 | yes | 
│ Huli Huli Chicken | AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72 | https://www.allrecipes.com/recipe/229690/huli-huli-chicken/ │ yes │
| 

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[20:25]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[25:30]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[31:35]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[35:40]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[41:45]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[45:50]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[50:55]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[55:60]
         )


In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[60:69]
         )


Need to test search queries that:
1) Get the recipe we're looking for
    - Query should already have the recipe title, recipe cuisine
    - We use the title and cuisine to get the ingredients
2) Do not return the same recipe name back
    eg `WHERE !(title contains [query_recipe_title])`
    or `WHERE !(title [fuzzy logic] [query_recipe_title])`
3) Do not return the same cuisine back
    eg `WHERE !(cuisine contains [query_recipe_cuisine or query_recipe_cuisine_superset])`
4) Maximize ingredient similarity (start with cosine similarity)
    - Could try vespa dotProduct, wand


In [ ]:
# ok baseline recipes
baselines = [
    "BBCFood-11015de4db20e53dcda0035673dfdbeca355d2a1d37d71f13d5fe5ea7ccca02d", #cacio e pepe
    "AllRecipes-fddb273d401ac45b568695daeb1abe0b632696dd867992fd401e31dc794d66f9", #lasagna
    "AllRecipes-715d84d02e183deaaf2400f7526c80a2b01a25032c1abb125c28fe4d879cc672", #margherita pizza
    "AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e", #tiramisu
    "AllRecipes-867b4f3b539b381fb9f35252402fc14efc9d6fcc64353089959ac159372a8199", #tiramisu
    "BBCFood-c6b9581aa49ee6b0cc3f686f58cfd124c395284cc46dfcde5bc931982975ee15", #tiramisu
    "BBCFood-d59ab715a7f6f0022155b1149e4e059b0497fd790993fb11f1795b901cae8f97", #cannoli
    "AllRecipes-d431cefd9f595be8a8fcd7ff9e05a82ca1b79220c72edde1aa2a0b6574060c04", #arancini
    "BBCFood-056f016b0b0d8c383c1a6bf47077a46a64529c04ab043c14dce3b23d3980e0ed", # pad thai
    "AllRecipes-d4bb95742a7fd71eb848a7d4b2a4dc13220e5bb378e1b6805d5ea493e0e1166d", # pad thai
    "Epicurious-0293b95bfc74cf90dc07e2ee399312e40e03d14dd99e5db84e678a1393b131cc", # pad thai
    "Epicurious-ca2d5e56121ae04e8b7bdcd07648796b2d95afed017fc56200535ad43ca8e51d", # kung pao chicken
    "Epicurious-7761b0f03aaf56b52b1de417213b66fddf1d49c554dabaacb23c9c7a260982ba", # boeuf bourguignon
    "Epicurious-e367b68ed6df8b0bccbeb73f9d82897294ef267fa3a74d30505ce7470b2384dc", # boeuf bourguignon 
    "AllRecipes-6914cfedf5d3af52ea66433959c958821b8940ee1948fbe29f841d48ad1de4a4", # beef wellington
    "AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23", # fish and chips
    "BBCFood-adcbfceaa8636f83ff8866d44bc7a44385c71250d73d427c08dbe345e4127143", # fish and chips
    "BBCFood-0a7c810d7b7c99ae728f9a2deff14340edb099949f20fed75a8083c29ca83aab", # chicken tikka masala
    "Epicurious-a0ab4c8e25f1fe8e65f400f93db23cf55e1e46cb4361cd65426f908f3013f33d", #chicken tikka masala
    "AllRecipes-bb6dd1b85cfd3cbfeb68160c187862ec6de84739123733ceeaae2976141f8936", # yorkshire pudding
    "AllRecipes-92c5c30f5c0b1655a1e1900fcdaa603917f2abf62529c130713412a844046e33", # yorkshire pudding
    "AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0", # apple pie
    "AllRecipes-384fd2f67b06a5bfdf532f94d5e89df99e08b91c5d9dace20fd29bba7b8fdad3", # apple pie, this might need to get taken out
    "Epicurious-aa7806301b6fc904134d692a2a425d0a0fd5a1f6f95cb431e8c31173db7d99fa", # apple pie
    "BBCFood-ee117ce84728c9b2e90654f59895a89dc7c10599b61bed83f55e9bdf723e6890", # scotch egg
    "BBCFood-53dc7bd8da3ef8a7dedfef0b2be1880f71585752d8113313c6d9e49bd5cba6fa", # scotch egg
    "BBCFood-2ab22cd0a3889f38cf740040d33aae5a4baa8568b70cde5233d6f8a673958140", # haggis
    "BBCFood-47410dc7097b2609d5c0a80e4ce6f938b00be18b52907e5a5322e33d6edee335", # buffalo wings
    "AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e", # banana pudding
    "Epicurious-ca99be627e88806d8357a5cbfdd314b3643028f275e6ae1c90adfcf3ae9e30d4", # cincinnati chili
    "Epicurious-7e9ff3fc13b040941ed9f7a4bbfd809e17a9ce7d5b569f1f6c4ba96e9a9ae140", #kalbi
    "Epicurious-4b6f818877199d375e801cec974c6f6b037b4c0b2aa2bbec0238dd77f797264d", # seafood pancake
    "AllRecipes-c6d0cd77e37ed8069f0d9e72f9e832026459273ac1bc2587800dc8fcd1418ec7", #kimchi soup
    "AllRecipes-48bde26e015dc9cbcbb540c4710248d538227d4afe5aa3e977e34710289789bb", # menudo
    "AllRecipes-cac0d29872bd9f49d34d8c9eac41a3806929420f84e36b8e579c3bce939cdc7b", # pozole
    "AllRecipes-9a5dc3fa2511284deeb5ee469d110874b0de9c021439cd0e088d3ef4f53f28bf", # carnitas
    "BBCFood-6d6cc18dadaffa5ea10194b59aa6e310d34b1f713756d0b528cc57c5e61d92e3", # carnitas
    "AllRecipes-5fc2fc72d7caf510b4ccde625c54a0907de1489d323d385d74c2a6c9c5d61c7b", # elotes
    "AllRecipes-8aa892f3f4d827f781d2bc1491351d982144e2fa2bb3226ebf5cc6af6df5ac5a", # mexican rice
    "AllRecipes-4f31ce862b67de4659f76e3a4de016a32770c878544d112aae95cc09b63d799a", # pho
    "Epicurious-b3f8d44e48bba1b47e5cf1482d8c5e076359fa919061bbfd2f075d131aea2025", # bagels
    "Epicurious-6a7b3ef6808832fbf1ab5a8e539eb794866d630c1b65218ee7b5c385b78f65a4", # yellow pea soup
    "AllRecipes-0caa9e3e35dee2910e6f81c869d1176a2de129b8e3c7fddf6e469b9e3977fadb", # custard 
    "AllRecipes-3c9cbb4d78d017a21a48907adbc31c2593ebbd0717b9391779a2f75b531fcbe4", # tourtiere
    "Epicurious-d2b862c91c84a69b9b2f2be1a1cc9702a2d6af8a6f1e5dcaccbacda6ed9f31b0", # california roll
    "AllRecipes-b8116cc920cd9d916cd196927ad2af1062785cdaa3d3b4fc3e997ee86fd75c84", # pierogi
    "Epicurious-4284e53129c7dc89f97f2f1933843946745df061370bd1ab738bf2afe1c537c3", # pierogi
    "AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72", # huli huli chicken
    "AllRecipes-bd864502bd1b052f501d3148d66effc966246a69a5e705c585fa366d9599e090", # huli huli chicken
    "AllRecipes-1b1ac4f279d0cbb0c680da7a1839d0031a52f3cc73681dbb2644cf96d6aec4f9", # huli huli chicken
]

In [ ]:
len(baselines)

In [ ]:
baseline_df = filtered_to_investigate.filter(
    pl.col("mealeon_id").str.contains_any(baselines)
)

baseline_df

In [ ]:
baseline_df.filter(
    pl.col("title").str.to_lowercase().str.contains("tiramisu")
)

In [ ]:
# polars search can be 
recipe_title = "tiramisu"
recipe_cuisine = "italian"

baseline_df.filter(
    pl.col("title").str.to_lowercase().str.contains(recipe_title),
    pl.col("cuisines").list.contains(recipe_cuisine.title())
)


- From this dataframe, we want the `mealeon_id`, `ingredients` to be used in Vespa

- `mealeon_id` can be used directly in vespa. JUST KIDDING

- `title`

-  `ingredients` need some form of vectorization/embedding creation

- embedding creation can be done with vespa feed?
  - might be able to use json feed 

- vespa distance-metric with the filters above for avoiding same name and same cuisine

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where title contains '{recipe_title}' limit 10",
        query=recipe_title,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()

In [ ]:
pprint.pp(response.hits)

In [ ]:
recipe_title

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f'select * from sources mealeon2 where !(title contains "{recipe_title}") and !(cuisine in {recipe_cuisine}) limit 10',
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f'select * from sources mealeon2 where !(cuisine in {recipe_cuisine}) limit 10',
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
type(recipe_cuisine)

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisine in {[recipe_cuisine]}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
recipe_cuisine

In [ ]:
print([recipe_cuisine])

In [ ]:
cuisines_queried = [recipe_cuisine]

with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisine in {cuisines_queried}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisine in ({recipe_cuisine}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisines in ('italian')) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
pprint.pp(response.hits)

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisines in ('italian')) and !(title contains '{recipe_title}') limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
pprint.pp(response.hits)

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisines in ('{recipe_cuisine}')) and !(title contains '{recipe_title}') limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
pprint.pp(response.hits)

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon2 where !(cuisines in ('{recipe_cuisine}')) or !(title contains '{recipe_title}') limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()
pprint.pp(response.hits)

In [ ]:
# actual results
results = resp_json['hits']
results

In [ ]:
# | hide
nbdev.nbdev_export()